# Escalamiento — inferencia del modelo afinado sobre todo el corpus

Aplica el modelo afinado con LoRA (mejor *fold*, `modelos/Modelo_fold_3`) a **todos** los comentarios del corpus para *escalar* la clasificación más allá de los 581 etiquetados a mano.

> Antes de ejecutar: **Restart** del kernel y luego *Run All*, para no arrastrar memoria de corridas previas.

## 1. Configuración: rutas, GPU/CPU y librerías

In [26]:
import os
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel
warnings.filterwarnings('ignore')

# ====== AJUSTA ESTO EN TU MÁQUINA ======
# Ruta a la raíz del repositorio LLM_PROJECT_1:
PROJECT_ROOT = Path(os.getcwd()).parent
USAR_GPU   = True   # ponlo en False si tu GPU tiene poca memoria (p. ej. 4 GB)
BATCH_SIZE = 8      # tamaño de lote; bájalo (4) o súbelo (16/32) según tu memoria
# =======================================

device = 'cuda' if (USAR_GPU and torch.cuda.is_available()) else 'cpu'
semilla = 61298
np.random.seed(semilla)

print("PROJECT_ROOT:", PROJECT_ROOT, "| existe:", PROJECT_ROOT.exists())
print("Dispositivo de inferencia:", device)
if device == 'cuda':
    print("GPU:", torch.cuda.get_device_name(0))

PROJECT_ROOT: c:\Users\nicte\Documents\LLM_PROJECT_1 | existe: True
Dispositivo de inferencia: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


## 2. Carga del corpus

Se carga el corpus unificado quedándose con `comentario`, `etiquetado_humano` y `categoria`. La inferencia se hará sobre **todos** los comentarios (no solo los etiquetados).

In [27]:
MODELO_TUNEADO_PATH = PROJECT_ROOT / 'modelos' / 'Modelo_fold_3'   # ojo: minúscula 'fold'
DATOS_PATH = PROJECT_ROOT / 'data' / 'limpieza_final' / 'etiquetado_humano_unificado.csv'

corpus = pd.read_csv(DATOS_PATH, usecols=['comentario', 'etiquetado_humano', 'categoria'],
                     encoding='utf-8-sig')
print("Comentarios en el corpus:", len(corpus))
corpus.head()

Comentarios en el corpus: 2578


,comentario,etiquetado_humano,categoria
0,cuál es el más cercado para rayar el nombre de...,3.0,infraestructura
1,esos baños deberian estar en el metro no saben...,2.0,infraestructura
2,no pues bueno me da risa,3.0,infraestructura
3,los van a abandonar,2.0,infraestructura
4,nada los tiene contentos,4.0,infraestructura


## 3. Carga del modelo base + adaptadores LoRA

Se reconstruye el modelo base `nlptown/bert-base-multilingual-uncased-sentiment` y se le acoplan los adaptadores LoRA entrenados (modo `local_files_only`). Luego se carga el *tokenizador* y se mueve todo al dispositivo elegido.

In [28]:
modelo = 'nlptown/bert-base-multilingual-uncased-sentiment'

clasificador = AutoModelForSequenceClassification.from_pretrained(
    modelo,
    num_labels=5,
    id2label={0: 'Negativo', 1: 'Parcialmente Negativo', 2: 'Neutral',
              3: 'Parcialmente Positivo', 4: 'Positivo'},
    label2id={'Negativo': 0, 'Parcialmente Negativo': 1, 'Neutral': 2,
              'Parcialmente Positivo': 3, 'Positivo': 4},
)

clasificador_tuneado = PeftModel.from_pretrained(
    clasificador, str(MODELO_TUNEADO_PATH),
    local_files_only=True, ignore_mismatched_sizes=True,
)
tokenizador = AutoTokenizer.from_pretrained(str(MODELO_TUNEADO_PATH))
clasificador_tuneado = clasificador_tuneado.to(device)
clasificador_tuneado.eval()
print("Clasificador tuneado listo en:", device)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Clasificador tuneado listo en: cuda


## 4. Inferencia por MINI-LOTES 

En cada paso:
- se tokeniza solo el lote,
- se obtienen clase predicha y confianza,
- se pasan los resultados a CPU y se **libera la memoria** del lote (`empty_cache`).

In [29]:
comentarios = corpus['comentario'].astype(str).tolist()
resultados = []

with torch.no_grad():
    for inicio in range(0, len(comentarios), BATCH_SIZE):
        lote = comentarios[inicio:inicio + BATCH_SIZE]

        tok = tokenizador(lote, return_tensors='pt', truncation=True,
                          max_length=512, padding=True)
        tok = {k: v.to(device) for k, v in tok.items()}

        logits = clasificador_tuneado(**tok).logits
        probabilidades = torch.softmax(logits, dim=-1)
        predicciones = torch.argmax(logits, dim=-1)

        for j, texto in enumerate(lote):
            resultados.append({
                'comentario': texto,
                'etiqueta estimada': int(predicciones[j].item()),
                'confianza': float(torch.max(probabilidades[j]).item()),
            })

        del tok, logits, probabilidades, predicciones
        if device == 'cuda':
            torch.cuda.empty_cache()

print(f"Inferencia completa sobre {len(resultados)} comentarios (lotes de {BATCH_SIZE})")

Inferencia completa sobre 2578 comentarios (lotes de 8)


## 5. Tabla de resultados y guardado

Se construye el `DataFrame` con la etiqueta estimada (convertida de 0-4 a **1-5**), la etiqueta humana (queda `NaN` en los comentarios sin etiquetar) y la confianza.

In [30]:
res_df = pd.DataFrame({
    'comentario': corpus['comentario'].values,
    'categoria': corpus['categoria'].values,
    'etiqueta_estimada': [r['etiqueta estimada'] + 1 for r in resultados],  # 0-4 -> 1-5
    'etiqueta_real': corpus['etiquetado_humano'].values,
    'confianza': [r['confianza'] for r in resultados],
})

salida = PROJECT_ROOT / 'data'/ 'resultados'/ 'escalamiento_resultados_v2.csv'
res_df.to_csv(salida, index=False, encoding='utf-8-sig')
print("Guardado en:", salida)
res_df.head(10)

Guardado en: c:\Users\nicte\Documents\LLM_PROJECT_1\data\resultados\escalamiento_resultados_v2.csv


,comentario,categoria,etiqueta_estimada,etiqueta_real,confianza
0,cuál es el más cercado para rayar el nombre de...,infraestructura,3,3.0,0.575264
1,esos baños deberian estar en el metro no saben...,infraestructura,2,2.0,0.576533
2,no pues bueno me da risa,infraestructura,2,3.0,0.489879
3,los van a abandonar,infraestructura,2,2.0,0.559644
4,nada los tiene contentos,infraestructura,3,4.0,0.359283
5,se ve como 1520 no se de que te quejas,infraestructura,4,4.0,0.506070
6,no maaaa,infraestructura,2,2.0,0.490568
7,y que volteas hacia arriba,infraestructura,2,3.0,0.516658
8,me da risa exacto hay un montón de fugas porqu...,infraestructura,2,2.0,0.508893
9,que pensaban la lamparita costó bastantes mill...,infraestructura,2,2.0,0.685036
